# KaoLRM — Milestone 1 Geometry Bake-Off 비교 (Colab H100)

Pixel3DMM와 **동일한 입력**으로 비교할 2026 후보. FLAME params + mesh + colored Gaussians 출력.

- 계획: `experiments/milestone1_geometry_bakeoff/README.md`
- **License: 소스 Apache 2.0이나 EG3D/FLAME/weights 때문에 effective use는 비상업 연구.**

> ⚠️ Pixel3DMM과 **다른 conda env / 다른 Colab runtime**에서 실행한다 (torch 버전 충돌 방지).
> ⚠️ private 입력/출력은 git 금지. 명령은 2026-06-21 공식 README 기준.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. conda 설치 (condacolab) — 실행 후 커널 재시작

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()

## 2. clone + 환경

In [ ]:
import condacolab; condacolab.check()
import os
%cd /content
if not os.path.exists('/content/KaoLRM'):
    !git clone https://github.com/CyberAgentAILab/KaoLRM.git
%cd /content/KaoLRM
!git rev-parse HEAD

In [ ]:
%%bash
set -e
conda create -n kaolrm python=3.10 -y
conda run -n kaolrm pip install torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu126
cd /content/KaoLRM
conda run -n kaolrm pip install --no-build-isolation -r requirements.txt
conda run -n kaolrm pip install xformers==0.0.33.post2 --index-url https://download.pytorch.org/whl/cu126

## 3. FLAME 다운로드

`fetch_data.sh`는 https://flame.is.tue.mpg.de 계정/동의가 필요하다.

In [ ]:
%%bash
set -e
cd /content/KaoLRM
conda run -n kaolrm bash fetch_data.sh

## 4. 사전학습 checkpoint 배치

Releases 페이지에서 받아 `releases/mono/`(frontal), `releases/multiview/`(profile)에 둔다.
TODO: Releases 자산 URL을 확인해 아래에 채운다.
https://github.com/CyberAgentAILab/KaoLRM/releases

In [ ]:
%%bash
set -e
cd /content/KaoLRM
mkdir -p releases/mono releases/multiview
# TODO: Releases에서 checkpoint 다운로드 후 위 폴더에 배치
# 예: wget <RELEASE_ASSET_URL> -O releases/mono/<file>
ls -R releases

## 5. private 입력 준비 (배경 제거)

KaoLRM은 배경 제거된 이미지(OpenLRM 규약, rembg/Clipdrop)를 받는다.
Pixel3DMM과 **같은 원본 세트**에서 동일 프레임을 사용한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# TODO: 본인 입력 세트 경로로 교체 (Pixel3DMM과 동일 원본)
INPUT_DIR = '/content/drive/MyDrive/hair_app_private/m1_inputs/set01'
import os; print('exists:', os.path.exists(INPUT_DIR))

In [ ]:
%%bash -s "$INPUT_DIR"
set -e
cd /content/KaoLRM
conda run -n kaolrm pip install rembg
mkdir -p /content/kao_inputs_nobg
for f in "$1"/*; do
  conda run -n kaolrm rembg i "$f" "/content/kao_inputs_nobg/$(basename "${f%.*}").png" || true
done
ls /content/kao_inputs_nobg

## 6. 추론

frontal/in-the-wild = `infer_mono.sh`, profile = `infer_multiview.sh`.
TODO: 두 스크립트의 입력 경로 인자를 `/content/kao_inputs_nobg` 로 맞춘다 (스크립트 내부 변수 확인).

In [ ]:
%%bash
set -e
cd /content/KaoLRM
# TODO: 스크립트가 입력/출력 경로를 인자로 받는지 확인 후 조정
conda run -n kaolrm sh infer_mono.sh
# conda run -n kaolrm sh infer_multiview.sh   # profile views

## 7. 출력 확인 / 렌더

출력은 `dumps/releases/{model_type}/` 에 .ply mesh, .npy FLAME params, .png/.gif.
Pixel3DMM과 **동일 camera·neutral material**로 렌더해 비교한다.

In [ ]:
import glob
for p in sorted(glob.glob('/content/KaoLRM/dumps/**/*', recursive=True))[:50]:
    print(p)

## 8. Run manifest 기록

In [ ]:
import json, subprocess, datetime, pathlib
commit = subprocess.run(['git','-C','/content/KaoLRM','rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
manifest = {
    'model': 'kaolrm',
    'commit': commit,
    'license': 'Apache-2.0 code; effective non-commercial (EG3D/FLAME/weights)',
    'input_set_id': 'set01',
    'mode': 'mono',   # or multiview
    'gpu': 'H100',
    'created_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'fixes_applied': [],
}
dst = pathlib.Path('/content/drive/MyDrive/hair_app_private/m1_manifests')
dst.mkdir(parents=True, exist_ok=True)
out = dst / 'kaolrm_set01.json'
out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 9. 점수화

`scoring_sheet.csv`에 Pixel3DMM과 같은 기준으로 기록하고, README Gate에 따라 임시 baseline을 고른다.